# 15 · Malha em operação: treinar com o passado e prever os anos seguintes

> **Regra da disciplina (aula 3):** especificação FOCUS → código gerado pelo assistente → executar → *gate* contra uma resposta já conhecida → aceitar.
> **Quando o gate falha, a restrição que faltou entra em `U`. O código não é remendado.**
>
> Cada passo abaixo tem três partes: a especificação FOCUS enviada ao assistente, o gate declarado **antes** do código (com a fonte da resposta conhecida e da tolerância) e o código que voltou, executado e gateado. O registro de uso de IA é exportado ao final (`registro_uso_ia.csv`).

**Foco definido pelo autor (16/11).** O uso principal é a malha existente: com o que se sabe até um ano T (atributos da via e histórico de acidentes), apontar os quilômetros que terão descarrilamento por via nos anos seguintes, inclusive os que ainda não tiveram nenhum.

**Desenho (declarado antes dos resultados).** Janela móvel, sem olhar o futuro: para cada ano de corte T de 2016 a 2022, o modelo é ajustado com o histórico até T−3 para explicar os descarrilamentos de T−2 a T, e depois aplicado com o histórico até T para prever T+1 a T+3. São sete cortes; nenhum dado do período previsto entra no ajuste nem nos atributos. Métricas na malha inteira e também só nos quilômetros que nunca tinham tido descarrilamento por via até T.

**Métodos comparados em todos os cortes.** Regra do EAR (sem ajuste); histórico do próprio quilômetro; histórico do quilômetro, da vizinhança, da linha e da malha (logística); atributos da via sem histórico (modelo M2); atributos mais histórico (logística e gradient boosting, hiperparâmetros fixos).

**Gates.** 2.681 descarrilamentos por via entre 2011 e 2025; janelas de ajuste e de teste disjuntas em todos os cortes; nenhum atributo usa ano posterior a T.

In [1]:
# === Setup: caminhos, bibliotecas e as três ferramentas (gate, gate_between, log_ia) ===
import os, sys, json, re, unicodedata, datetime as dt
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if os.environ.get("ET261400_BASE"):
    BASE = os.environ["ET261400_BASE"]                # definido pelo notebook 00 (execução em lote) ou pela execução local
elif IN_COLAB and os.path.exists("/content/ET261400/data"):
    BASE = "/content/ET261400"                        # pacote já descompactado na sessão do Colab
elif IN_COLAB and os.path.exists("/content/et261400_colab.zip"):
    import zipfile
    zipfile.ZipFile("/content/et261400_colab.zip").extractall("/content/ET261400")   # pacote enviado pelo painel de arquivos
    BASE = "/content/ET261400"
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE = "/content/drive/MyDrive/ET261400"          # pasta do projeto no Drive (ajustar se for outra)
else:
    BASE = os.environ.get("ET261400_BASE", os.path.abspath(".."))   # execução local: raiz do repositório
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
FIG, TAB = f"{BASE}/results/figures", f"{BASE}/results/tables"
for d in (PROC, FIG, TAB):
    os.makedirs(d, exist_ok=True)
print("BASE =", BASE)

_GATES = []          # todo gate executado fica registrado aqui
_LOG_IA = []         # registro de uso de IA: passo, ferramenta, especificação, resultado

def gate(name, got, expected, tol=0.0, source=""):
    """Compara um número calculado com um número que já se tinha antes do código.
    tol vem da fonte (meio passo de arredondamento, faixa publicada), nunca é alargada até passar."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = abs(got - expected) <= tol
    _GATES.append(dict(gate=name, got=got, expected=expected, tol=tol, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, esperado {expected} (tol +/-{tol}) {('| ' + source) if source else ''}")
    return ok

def gate_between(name, got, low, high, source=""):
    """O mesmo, quando a fonte publica uma faixa e não um número."""
    if got is None:
        print(f"[ -- ] {name}: nada calculado ainda"); return False
    ok = low <= got <= high
    _GATES.append(dict(gate=name, got=got, expected=f"[{low}, {high}]", tol=None, ok=ok, source=source))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: obtido {got}, faixa [{low}, {high}] {('| ' + source) if source else ''}")
    return ok

def log_ia(passo, ferramenta, especificacao, resultado, verificacao):
    """Registro de uso de IA exigido pela disciplina: o que foi pedido, a que ferramenta, o que voltou e como foi verificado."""
    _LOG_IA.append(dict(data=dt.date.today().isoformat(), passo=passo, ferramenta=ferramenta,
                        especificacao=especificacao.strip(), resultado=resultado, verificacao=verificacao))

def exportar_registros(prefixo):
    pd.DataFrame(_GATES).to_csv(f"{TAB}/{prefixo}_gates.csv", index=False)
    pd.DataFrame(_LOG_IA).to_csv(f"{TAB}/{prefixo}_registro_uso_ia.csv", index=False)
    n_ok = sum(g["ok"] for g in _GATES)
    print(f"gates: {n_ok}/{len(_GATES)} PASS | registros de IA: {len(_LOG_IA)} | exportados em {TAB}")

def norm_txt(s):
    """Maiúsculas sem acento, espaços simples: chave de junção para nomes de pátios e estações."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).upper().strip()


BASE = /home/claude


In [2]:
import warnings; warnings.filterwarnings("ignore")
from scipy import sparse
from scipy.stats import rankdata
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
SEED, H = 42, 3
C0 = pd.read_csv(f"{PROC}/11_celulas_1km.csv")
C = C0[~C0["sentido"].astype(str).str.contains("Interromp")].sort_values(["ferrovia", "linha", "km_a"]).reset_index(drop=True); gate("células em operação", len(C), 20936, source="notebook 11")
E = pd.read_csv(f"{PROC}/11_eventos_celulas.csv"); E = E[E["cel_id"].isin(C["cel_id"])].copy(); E["ano"] = E["ano"].astype(int); EV = E[E["via"]]
gate("descarrilamentos por via nas células", len(EV), 2681, source="notebook 11"); gate_between("anos dos eventos", float(EV["ano"].min()), 2011, 2011, source="recorte 2011 a 2025"); gate_between("último ano dos eventos", float(EV["ano"].max()), 2025, 2025, source="recorte 2011 a 2025")
display(EV.groupby("ano").size().to_frame("descarrilamentos por via").T)
idx = pd.Series(np.arange(len(C)), index=C["cel_id"])
def contagem(ev, a0, a1):
    v = np.zeros(len(C)); s = ev[(ev["ano"] >= a0) & (ev["ano"] <= a1)].groupby("cel_id").size(); v[idx[s.index].values] = s.values; return v
DIST = ["dmin_patio", "dmin_amv", "frac_zona_patio"] + [f"f16p{j}" for j in range(5)] + [f"f16a{j}" for j in range(5)]
CURV = ["rmin_geo"] + [f"fr_lt{l}" for l in (300, 400, 500, 600, 800)]
NUM = ["rampa_max_pct", "carga_eixo_t", "taxa_dorm", "gab_vert_m", "gab_horiz_m", "cap_vinc", "dias_op", "raio_min_m", "vma_carregado"]
CAT = ["dormente", "perfil_trilho", "fixacao", "sinalizacao", "controle_circ", "bitola", "sentido", "n_linhas"]
for c in NUM + DIST + CURV: C[c] = pd.to_numeric(C[c], errors="coerce")
for c in CAT: C[c] = C[c].astype(str).str.strip().replace({"nan": "NA", "": "NA"})
def design(df, num, cat):
    X = df[num].copy(); med = X.median().fillna(0.0)
    for c in num: X[c + "_na"] = X[c].isna().astype(float); X[c] = X[c].fillna(med[c])
    return pd.concat([X, pd.get_dummies(df[cat], dtype=float)], axis=1).values
X0 = design(C, DIST + CURV + NUM, CAT); LN = C["len_km"].values; fer = np.asarray(C["ferrovia"], dtype=object); LIN = np.asarray(C["ferrovia"] + "|" + C["linha"].astype(str), dtype=object); SEG = np.asarray(C["seg_id"].astype(str), dtype=object); KM = ((C["km_a"] + C["km_b"]) / 2).values
ri, ci = [], []
for l in np.unique(LIN):
    i = np.where(LIN == l)[0]; k = KM[i]; M = (np.abs(k[:, None] - k[None, :]) <= 20) & ~np.eye(len(i), dtype=bool); a, b = np.where(M); ri.append(i[a]); ci.append(i[b])
W = sparse.csr_matrix((np.ones(sum(len(a) for a in ri)), (np.concatenate(ri), np.concatenate(ci))), shape=(len(C), len(C))); nviz = np.asarray(W.sum(1)).ravel()
def historico(T):
    """Atributos de histórico com dados de 2011 até o ano T, por ano de observação (comparáveis entre cortes)."""
    assert T <= 2025; n = T - 2010; hv = contagem(EV, 2011, T); ha = contagem(E, 2011, T); b = (hv > 0).astype(float); d = pd.DataFrame({"b": b, "hv": hv})
    return np.column_stack([hv / n, ha / n, b, (W @ b) / np.maximum(nviz, 1), (W @ hv) / np.maximum(nviz, 1) / n, d.groupby(SEG)["b"].transform("mean").values, d.groupby(LIN)["b"].transform("mean").values, d.groupby(fer)["b"].transform("mean").values, d.groupby(fer)["hv"].transform("mean").values / n]), b
def cls16(d): return np.select([d <= 300, d <= 600, d <= 1200, d <= 1800], [4, 3, 2, 1], 0)
def rcls16(r_): return np.select([r_ < 500, r_ < 1200, r_ < 2300], [3, 2, 1], 0)
B2 = (0.5 * cls16(C["dmin_patio"].fillna(1e9)) / 4 + 0.5 * rcls16(C["rmin_geo"].fillna(1e6)) / 3).astype(float)
def auc(yy, s, m=None):
    if m is not None: yy, s = yy[m], s[m]
    n1 = yy.sum(); n0_ = len(yy) - n1; return float((rankdata(s)[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0_)) if n1 >= 5 and n0_ >= 5 else np.nan
def auc_dentro(s, yy, m=None):
    a, w = [], []
    for f in np.unique(fer):
        i = (fer == f) if m is None else ((fer == f) & m)
        if yy[i].sum() >= 5 and yy[i].sum() < i.sum(): a.append(auc(yy[i], s[i])); w.append(yy[i].sum())
    return float(np.average(a, weights=w)) if a else np.nan
def captura(s, evv, p=0.20):
    o = np.lexsort((np.random.default_rng(SEED).random(len(s)), -s)); cum = np.cumsum(LN[o]) / LN.sum(); return float(evv[o][cum <= p].sum() / evv.sum())
def lr(Xtr, ytr, Xte):
    st = StandardScaler().fit(Xtr); return LogisticRegression(C=0.5, max_iter=3000).fit(st.transform(Xtr), ytr).predict_proba(st.transform(Xte))[:, 1]
def gbm(Xtr, ytr, Xte): return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0, early_stopping=False, random_state=SEED).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
log_ia("15.1 dados e histórico", "Claude", "FOCUS: histórico por ano de observação até T; vizinhança de 20 km na mesma linha; nada posterior a T", "historico(T), W", "4 gates de contagem e de período")

[PASS] células em operação: obtido 20936, esperado 20936 (tol +/-0.0) | notebook 11
[PASS] descarrilamentos por via nas células: obtido 2681, esperado 2681 (tol +/-0.0) | notebook 11
[PASS] anos dos eventos: obtido 2011.0, faixa [2011, 2011] | recorte 2011 a 2025
[PASS] último ano dos eventos: obtido 2025.0, faixa [2025, 2025] | recorte 2011 a 2025


ano,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
descarrilamentos por via,324,291,246,214,261,165,185,166,183,128,152,120,112,77,57


In [3]:
L = []
for T in range(2016, 2023):
    tr_a0, tr_a1, te_a0, te_a1 = T - H + 1, T, T + 1, T + H; assert tr_a1 < te_a0 and te_a1 <= 2025
    Htr, _ = historico(T - H); Hte, ja = historico(T); ytr = (contagem(EV, tr_a0, tr_a1) > 0).astype(int); cte = contagem(EV, te_a0, te_a1); yte = (cte > 0).astype(int); novo = ja == 0
    S = {"Regra do EAR (sem ajuste)": B2, "Histórico do próprio km": Hte[:, 0], "Histórico (km, vizinhança, linha, malha)": lr(Htr, ytr, Hte), "Atributos da via, sem histórico (M2)": lr(X0, ytr, X0),
         "Atributos + histórico (logística)": lr(np.column_stack([X0, Htr]), ytr, np.column_stack([X0, Hte])), "Atributos + histórico (gradient boosting)": gbm(np.column_stack([X0, Htr]), ytr, np.column_stack([X0, Hte]))}
    for nome, s in S.items():
        L.append(dict(corte=T, ajuste=f"{tr_a0} a {tr_a1}", previsto=f"{te_a0} a {te_a1}", metodo=nome, atingidas=int(yte.sum()), atingidas_sem_acidente_antes=int(yte[novo].sum()), auc_malha_inteira=auc(yte, s), auc_dentro=auc_dentro(s, yte), captura20=captura(s, cte), auc_km_sem_acidente_antes=auc(yte, s, novo), auc_dentro_km_sem_acidente_antes=auc_dentro(s, yte, novo)))
    print(T, "ok", flush=True)
R_ = pd.DataFrame(L); R_.to_csv(f"{TAB}/15_previsao_por_corte.csv", index=False)
gate("cortes avaliados", int(R_["corte"].nunique()), 7, source="desenho: T de 2016 a 2022")
res = R_.groupby("metodo", sort=False)[["auc_malha_inteira", "auc_dentro", "captura20", "auc_km_sem_acidente_antes", "auc_dentro_km_sem_acidente_antes"]].agg(["mean", "min", "max"]).round(3); display(res); res.to_csv(f"{TAB}/15_previsao_resumo.csv")
c1 = R_.drop_duplicates("corte"); print("por corte: células atingidas no período previsto e, delas, quantas não tinham acidente antes:"); display(c1[["corte", "previsto", "atingidas", "atingidas_sem_acidente_antes"]].T)
display(R_.pivot(index="corte", columns="metodo", values="auc_malha_inteira").round(3))
log_ia("15.2 janela móvel", "Claude", "FOCUS: sete cortes; ajuste no passado, previsão dos 3 anos seguintes; métricas na malha inteira e nos km sem acidente anterior", "15_previsao_*.csv", "gate de número de cortes; asserções de janelas disjuntas")
exportar_registros("15")

2016 ok


2017 ok


2018 ok


2019 ok


2020 ok


2021 ok


... (3 outputs left out)

por corte: células atingidas no período previsto e, delas, quantas não tinham acidente antes:


,0,6,12,18,24,30,36
corte,2016,2017,2018,2019,2020,2021,2022
previsto,2017 a 2019,2018 a 2020,2019 a 2021,2020 a 2022,2021 a 2023,2022 a 2024,2023 a 2025
atingidas,504,450,434,373,365,295,236
atingidas_sem_acidente_antes,409,354,337,282,259,193,152


metodo,Atributos + histórico (gradient boosting),Atributos + histórico (logística),"Atributos da via, sem histórico (M2)","Histórico (km, vizinhança, linha, malha)",Histórico do próprio km,Regra do EAR (sem ajuste)
corte,,,,,,
2016,0.765,0.766,0.746,0.745,0.566,0.599
2017,0.755,0.758,0.741,0.732,0.575,0.579
2018,0.771,0.775,0.751,0.758,0.577,0.584
2019,0.770,0.798,0.764,0.787,0.584,0.565
2020,0.804,0.828,0.799,0.805,0.606,0.570
2021,0.826,0.825,0.800,0.807,0.632,0.570
2022,0.840,0.828,0.805,0.799,0.635,0.569


gates: 5/5 PASS | registros de IA: 2 | exportados em /home/claude/results/tables


---
## Limites

Os atributos da via são os da Declaração de Rede de 2025 em todos os cortes. O registro do RAAF mudou de arquivo em dezembro de 2020, o que pode alterar a cobertura entre períodos; a AUC, por ser de ordenação, é pouco sensível ao nível geral. O histórico só existe em malha em operação; para ferrovia nova vale o cenário B do notebook 14.

**Declaração de uso de IA (regra da disciplina, três partes).** Ferramentas: Claude (Anthropic), usado pelo autor como assistente de programação a partir das especificações FOCUS registradas neste notebook, e Gemini do Colab quando indicado. Parte do trabalho: geração do código de cada passo e revisão do texto das células. Responsável pelo conteúdo: o autor, que definiu o problema, as fontes, as especificações, os gates e a verificação, e executou o notebook do zero antes de reportar qualquer número.